# Cross-case 2D--3D particle-heterogeneity agreement (cached)

This plotting notebook performs no Voronoï calculation and does not open particle trajectories. For each configured 3D case it reads only the compact cache written by the final section of `analyze_flow_particle_coupling.ipynb`:

`z.flow_postprocessing/results/<3D case>/metrics/particle_heterogeneity_2D_vs_3D_selected_snapshots.nc`

Particle panels are discovered automatically from the cached `particle_label` metadata originally calculated in `run_particles.ipynb`. No particle-class list is hard-coded. Set `PARTICLE_LABELS` only when a subset is desired.

The cache also contains the paired 2D/3D heterogeneity at the selected snapshots and a copy of the domain-characterization JSON, so the dots can still be coloured by the same variables as the cross-case submesoscale-activity notebook.


In [ ]:
# ============================================================
# 1. IMPORTS AND PROJECT PATHS
# ============================================================

from pathlib import Path
import sys
import json
import importlib
import re
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from matplotlib.colors import Normalize, LogNorm
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D


# Notebook location:
# THESIS/z.parcels_postprocessing/notebooks/compare_particle_heterogeneity.ipynb
NB_DIR = Path.cwd().resolve()
PROJECT_DIR = NB_DIR.parent.parent
PARCELS_DIR = PROJECT_DIR / "z.parcels_postprocessing"
FLOW_DIR = PROJECT_DIR / "z.flow_postprocessing"
FLOW_SCRIPTS_DIR = FLOW_DIR / "scripts"

PARCELS_RESULTS_ROOT = (PARCELS_DIR / "results").resolve()
FLOW_RESULTS_ROOT = (FLOW_DIR / "results").resolve()

for path in [PROJECT_DIR, FLOW_SCRIPTS_DIR, PARCELS_DIR]:
    path = str(path)
    if path not in sys.path:
        sys.path.insert(0, path)

import theme.plot_theme as ptheme
import shcherbina_utils as shu

importlib.reload(ptheme)
ptheme.apply_theme()

importlib.reload(shu)
shu.apply_plot_style()


def apply_cross_case_style():
    """Use the exact font sizing applied in the activity comparison."""
    shu.apply_plot_style()
    plt.rcParams.update({
        "font.size": max(getattr(ptheme, "FONT_SIZE", 14), 19),
        "axes.titlesize": max(getattr(ptheme, "TITLE_SIZE", 20), 24),
        "axes.labelsize": max(getattr(ptheme, "LABEL_SIZE", 18), 22),
        "xtick.labelsize": max(getattr(ptheme, "TICK_SIZE", 16), 18),
        "ytick.labelsize": max(getattr(ptheme, "TICK_SIZE", 16), 18),
        "legend.fontsize": max(getattr(ptheme, "LEGEND_SIZE", 15), 17),
        "figure.titlesize": max(getattr(ptheme, "SUPTITLE_SIZE", 24), 27),
    })


apply_cross_case_style()

print(f"Notebook dir        : {NB_DIR}")
print(f"Particle results    : {PARCELS_RESULTS_ROOT}")
print(f"Flow results        : {FLOW_RESULTS_ROOT}")


In [ ]:
# ============================================================
# 2. USER SETTINGS
# ============================================================

CASE_NAMES = [
    "run_aug1",
    "run_aug1_f1",
    "run_aug1_f2",
    "run_aug2",
    "run_aug3",
    "run_dec1",
    "run_dec2",
    "run_feb1",
    "run_jan1",
    "run_jul1",
    "run_jun1",
    "run_mar1",
    "run_may1",
    "run_nov1",
    "run_oct1",
    "run_sep1",
]

# Particle panels are discovered from the particle_label values stored in each
# cache. Leave this as None to plot every available label. To plot only a
# subset, provide exact labels copied from run_particles.ipynb, for example:
# PARTICLE_LABELS = ["Passive particles"]
PARTICLE_LABELS = None

HETEROGENEITY_CACHE_FILENAME = (
    "particle_heterogeneity_2D_vs_3D_selected_snapshots.nc"
)

# Change only this setting to colour by another characteristic stored in the
# cached domain-characterization JSON.
# COLOR_VARIABLE = "epsilon_proxy_m2_s3"
COLOR_VARIABLE = "eke_mean_m2_s2"

# "auto" uses logarithmic colour normalization for epsilon, EKE and Ri.
COLOR_SCALE = "auto"  # "auto", "linear", or "log"
COLOR_MAP = "viridis"

COLOR_LABELS = {
    "glorys_mld_median_m": "GLORYS median MLD [m]",
    "mld_median_m": "MITgcm median MLD [m]",
    "epsilon_proxy_m2_s3": r"$\epsilon$ proxy [m$^2$ s$^{-3}$]",
    "eke_mean_m2_s2": r"Mean EKE [m$^2$ s$^{-2}$]",
    "Ri_b_median": r"Median bulk $Ri_b$",
    "ow_ett.epsilon_eddy_median_m2_s3":
        r"Median eddy $\epsilon$ [m$^2$ s$^{-3}$]",
}

HETEROGENEITY_AXIS_SCALE = "linear"  # "linear" or "log"
HETEROGENEITY_LIMITS = None

MARKERS = ["o", "s", "^", "D", "P", "X", "v", "<", ">"]

ANNOTATE_CASE_NAMES = False
CONNECT_CASE_SNAPSHOTS = False
SKIP_MISSING_CASES = True

SAVE_FIGURES = True
SAVE_SUMMARY_CSV = True

OUTPUT_DIR = (
    PARCELS_RESULTS_ROOT
    / "_comparisons"
    / "cross_case_particle_heterogeneity"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_CSV_PATH = (
    OUTPUT_DIR
    / f"particle_heterogeneity_2D_vs_3D_{COLOR_VARIABLE.replace('.', '_')}.csv"
)

if HETEROGENEITY_AXIS_SCALE not in {"linear", "log"}:
    raise ValueError("HETEROGENEITY_AXIS_SCALE must be 'linear' or 'log'.")

print(f"Colour variable : {COLOR_VARIABLE}")
print(f"Output directory: {OUTPUT_DIR}")


In [ ]:
# ============================================================
# 3. LIGHTWEIGHT CACHE-LOADING HELPERS
# ============================================================


def get_nested_value(mapping, key):
    value = mapping
    for part in key.split("."):
        if not isinstance(value, dict) or part not in value:
            raise KeyError(key)
        value = value[part]
    return value


def _normalise_text(value):
    return " ".join(str(value).split()).casefold() if value is not None else ""


def _as_text(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")
    return str(value)


def find_heterogeneity_cache(case_name):
    path = (
        FLOW_RESULTS_ROOT
        / case_name
        / "metrics"
        / HETEROGENEITY_CACHE_FILENAME
    )
    if not path.exists():
        raise FileNotFoundError(
            f"No cached heterogeneity file found for {case_name!r}:\n{path}\n"
            "Run the final heterogeneity-cache section of "
            "analyze_flow_particle_coupling.ipynb first."
        )
    return path


def _cached_particle_identity(ds, index):
    def read(name, fallback=""):
        if name not in ds:
            return fallback
        return _as_text(np.asarray(ds[name].values)[index])

    particle_key = read("particle_key")
    panel_label = read("particle_panel_label")
    source_label_3d = read("source_particle_label_3d")
    source_label_2d = read("source_particle_label_2d")

    # New caches use the exact run-metadata label as particle_key. These
    # fallbacks keep the reader compatible with older cache files.
    resolved_label = (
        panel_label
        or source_label_3d
        or source_label_2d
        or particle_key
    )
    resolved_key = particle_key or resolved_label

    return {
        "particle_key": resolved_key,
        "particle_label": resolved_label,
        "particle_panel_label": panel_label,
        "source_particle_tag_3d": read("source_particle_tag_3d"),
        "source_particle_tag_2d": read("source_particle_tag_2d"),
        "source_particle_label_3d": source_label_3d,
        "source_particle_label_2d": source_label_2d,
        "source_particle_class_3d": read("source_particle_class_3d"),
        "source_particle_class_2d": read("source_particle_class_2d"),
        "source_trajectory_3d": read("source_trajectory_3d"),
        "source_trajectory_2d": read("source_trajectory_2d"),
    }


def _selected_cached_particle_indices(ds, selected_labels=None):
    identities = [
        _cached_particle_identity(ds, index)
        for index in range(ds.sizes["particle_class"])
    ]

    if selected_labels is None:
        return list(enumerate(identities))

    requested = {_normalise_text(label) for label in selected_labels}
    selected = [
        (index, identity)
        for index, identity in enumerate(identities)
        if _normalise_text(identity["particle_label"]) in requested
    ]

    found = {
        _normalise_text(identity["particle_label"])
        for _, identity in selected
    }
    missing = requested - found
    if missing:
        available = [identity["particle_label"] for identity in identities]
        raise KeyError(
            "Requested particle labels were not found in the cache.\n"
            f"Missing normalized labels: {sorted(missing)}\n"
            f"Available labels: {available}"
        )

    return selected


def read_cached_case_records(
    case_name,
    color_variable,
    selected_labels=None,
):
    cache_path = find_heterogeneity_cache(case_name)

    with xr.open_dataset(cache_path) as ds_open:
        ds = ds_open.load()

    case_3d = str(ds.attrs.get("case_3d", case_name))
    case_2d = str(ds.attrs.get("case_2d", f"{case_name}_2D"))
    run_title = str(ds.attrs.get("run_title_info", case_name))

    characterization_raw = ds.attrs.get("domain_characterization_json", "{}")
    try:
        characterization = json.loads(characterization_raw)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"Invalid domain_characterization_json in {cache_path}"
        ) from exc

    color_value = float(get_nested_value(characterization, color_variable))

    snapshot_values = np.asarray(ds["snapshot"].values, dtype=int)
    time_days = np.asarray(ds["target_time_days_3d"].values, dtype=float)

    records = []
    for p_index, identity in _selected_cached_particle_indices(
        ds,
        selected_labels=selected_labels,
    ):
        for s_index, (snapshot, day) in enumerate(
            zip(snapshot_values, time_days)
        ):
            records.append({
                "case_3d": case_3d,
                "case_2d": case_2d,
                "run_title_info": run_title,
                "snapshot": int(snapshot),
                "time_days": float(day),
                "time_days_3d": float(ds["target_time_days_3d"].values[s_index]),
                "time_days_2d": float(ds["target_time_days_2d"].values[s_index]),
                "requested_obs_index_3d": int(
                    ds["requested_obs_index_3d"].values[s_index]
                ),
                "requested_obs_index_2d": int(
                    ds["requested_obs_index_2d"].values[s_index]
                ),
                "matched_obs_index_3d": int(
                    ds["matched_obs_index_3d"].values[p_index, s_index]
                ),
                "matched_obs_index_2d": int(
                    ds["matched_obs_index_2d"].values[p_index, s_index]
                ),
                "matched_time_days_3d": float(
                    ds["matched_time_days_3d"].values[p_index, s_index]
                ),
                "matched_time_days_2d": float(
                    ds["matched_time_days_2d"].values[p_index, s_index]
                ),
                "match_method_3d": _as_text(
                    ds["match_method_3d"].values[p_index, s_index]
                ),
                "match_method_2d": _as_text(
                    ds["match_method_2d"].values[p_index, s_index]
                ),
                "heterogeneity_3d": float(
                    ds["heterogeneity_3d"].values[p_index, s_index]
                ),
                "heterogeneity_2d": float(
                    ds["heterogeneity_2d"].values[p_index, s_index]
                ),
                "sigma_area_norm_3d": float(
                    ds["sigma_area_norm_3d"].values[p_index, s_index]
                ),
                "sigma_area_norm_2d": float(
                    ds["sigma_area_norm_2d"].values[p_index, s_index]
                ),
                "n_valid_voronoi_cells_3d": int(
                    ds["n_valid_voronoi_cells_3d"].values[p_index, s_index]
                ),
                "n_valid_voronoi_cells_2d": int(
                    ds["n_valid_voronoi_cells_2d"].values[p_index, s_index]
                ),
                **identity,
                "color_variable": color_variable,
                "color_value": color_value,
                "cache_file": str(cache_path),
                "selected_snapshot_file": str(
                    ds.attrs.get("selected_snapshot_file", "")
                ),
                "characterization_file": str(
                    ds.attrs.get("domain_characterization_file", "")
                ),
            })

    return records, case_2d, cache_path


In [ ]:
# ============================================================
# 4. LOAD COMPACT CACHES ONLY
# ============================================================

records = []
skipped_cases = []

for case_name in CASE_NAMES:
    try:
        case_records, case_2d, cache_path = read_cached_case_records(
            case_name,
            COLOR_VARIABLE,
            selected_labels=PARTICLE_LABELS,
        )
        records.extend(case_records)
        n_labels = len({record["particle_label"] for record in case_records})
        print(
            f"Loaded {case_name} -> {case_2d}: "
            f"{len(case_records)} cached points for {n_labels} labels "
            f"from {cache_path.name}"
        )
    except Exception as exc:
        message = f"{case_name}: {type(exc).__name__}: {exc}"
        if SKIP_MISSING_CASES:
            skipped_cases.append(message)
            warnings.warn(message)
        else:
            raise

comparison_df = pd.DataFrame(records)

if comparison_df.empty:
    raise RuntimeError("No valid cached case data were loaded.")

comparison_df = (
    comparison_df
    .sort_values(["particle_label", "case_3d", "time_days"])
    .reset_index(drop=True)
)

particle_catalog = (
    comparison_df[["particle_key", "particle_label"]]
    .drop_duplicates()
    .sort_values("particle_label")
    .reset_index(drop=True)
)

# Guard against one cache key being associated with conflicting labels.
label_count_by_key = (
    particle_catalog.groupby("particle_key")["particle_label"].nunique()
)
if (label_count_by_key > 1).any():
    raise RuntimeError(
        "A cached particle_key is associated with multiple labels:\n"
        f"{label_count_by_key[label_count_by_key > 1]}"
    )

print("\nParticle labels discovered from the caches:")
for label in particle_catalog["particle_label"]:
    print(f"  - {label}")

display(comparison_df[[
    "particle_label",
    "case_3d",
    "case_2d",
    "time_days",
    "heterogeneity_2d",
    "heterogeneity_3d",
    "color_value",
    "match_method_2d",
    "match_method_3d",
]])

if skipped_cases:
    print("\nSkipped cases:")
    for message in skipped_cases:
        print(f"  - {message}")

if SAVE_SUMMARY_CSV:
    comparison_df.to_csv(SUMMARY_CSV_PATH, index=False)
    print(f"Saved: {SUMMARY_CSV_PATH}")


In [ ]:
# ============================================================
# 5. CROSS-CASE 2D--3D HETEROGENEITY AGREEMENT PLOTS
# ============================================================


def infer_color_scale(variable, requested_scale):
    if requested_scale != "auto":
        return requested_scale

    name = variable.lower()
    return (
        "log"
        if any(token in name for token in ["epsilon", "eke", "ri_"])
        else "linear"
    )


def make_color_norm(color_values):
    color_scale = infer_color_scale(COLOR_VARIABLE, COLOR_SCALE)
    color_values = np.asarray(color_values, dtype=float)

    if color_scale == "log":
        if np.any(color_values <= 0.0):
            raise ValueError(
                f"{COLOR_VARIABLE} contains non-positive values."
            )
        lower = float(color_values.min())
        upper = float(color_values.max())
        if np.isclose(lower, upper):
            lower /= 1.01
            upper *= 1.01
        return LogNorm(lower, upper)

    if color_scale == "linear":
        lower = float(color_values.min())
        upper = float(color_values.max())
        if np.isclose(lower, upper):
            padding = max(abs(lower) * 0.01, 1.0e-12)
            lower -= padding
            upper += padding
        return Normalize(lower, upper)

    raise ValueError("COLOR_SCALE must be auto, linear, or log.")


def infer_heterogeneity_limits(data):
    if HETEROGENEITY_LIMITS is not None:
        lower, upper = map(float, HETEROGENEITY_LIMITS)
        if not lower < upper:
            raise ValueError("HETEROGENEITY_LIMITS must satisfy lower < upper.")
        if HETEROGENEITY_AXIS_SCALE == "log" and lower <= 0.0:
            raise ValueError("Logarithmic limits must be strictly positive.")
        return lower, upper

    values = np.concatenate([
        data["heterogeneity_2d"].to_numpy(dtype=float),
        data["heterogeneity_3d"].to_numpy(dtype=float),
    ])
    values = values[np.isfinite(values)]
    if values.size == 0:
        raise ValueError("No finite heterogeneity values are available.")

    minimum = float(values.min())
    maximum = float(values.max())

    if HETEROGENEITY_AXIS_SCALE == "log":
        if minimum <= 0.0:
            raise ValueError(
                "Logarithmic heterogeneity axes require positive values."
            )
        if np.isclose(minimum, maximum):
            return minimum / 1.05, maximum * 1.05
        padding_factor = np.exp(0.06 * np.log(maximum / minimum))
        return minimum / padding_factor, maximum * padding_factor

    span = maximum - minimum
    padding = 0.07 * span if span > 0.0 else max(abs(maximum) * 0.07, 0.1)
    return minimum - padding, maximum + padding


def particle_slug(particle_key):
    slug = re.sub(r"[^A-Za-z0-9_-]+", "_", str(particle_key))
    return slug.strip("_") or "particle"


def plot_cross_case_heterogeneity(data, particle_key, particle_label):
    apply_cross_case_style()

    data = data.loc[
        data["particle_key"] == particle_key
    ].copy()

    finite = np.isfinite(
        data[[
            "heterogeneity_2d",
            "heterogeneity_3d",
            "color_value",
        ]]
    ).all(axis=1)
    data = data.loc[finite].copy()

    if data.empty:
        raise RuntimeError(
            f"No finite values for particle label {particle_label!r}."
        )

    lower, upper = infer_heterogeneity_limits(data)
    data["x_plot"] = data["heterogeneity_2d"].clip(lower, upper)
    data["y_plot"] = data["heterogeneity_3d"].clip(lower, upper)
    data["clipped"] = (
        (data["heterogeneity_2d"] < lower)
        | (data["heterogeneity_3d"] < lower)
        | (data["heterogeneity_2d"] > upper)
        | (data["heterogeneity_3d"] > upper)
    )

    norm = make_color_norm(data["color_value"].to_numpy(dtype=float))
    cmap = plt.get_cmap(COLOR_MAP)
    days = sorted(data["time_days"].unique())

    if len(days) > len(MARKERS):
        raise ValueError("Add more marker definitions.")

    marker_by_day = {
        day: MARKERS[index]
        for index, day in enumerate(days)
    }

    # Figure dimensions and styling are identical to the activity comparison.
    fig, ax = plt.subplots(figsize=(10.8, 9.2), facecolor="white")
    ax.set_facecolor("white")

    if CONNECT_CASE_SNAPSHOTS:
        for _, case_data in data.groupby("case_3d"):
            case_data = case_data.sort_values("time_days")
            ax.plot(
                case_data["x_plot"],
                case_data["y_plot"],
                color="0.70",
                lw=1.0,
                alpha=0.45,
                zorder=1,
            )

    for day in days:
        subset = data[data["time_days"] == day]
        edgecolors = [
            "black" if clipped else "white"
            for clipped in subset["clipped"]
        ]

        ax.scatter(
            subset["x_plot"],
            subset["y_plot"],
            c=subset["color_value"],
            cmap=cmap,
            norm=norm,
            marker=marker_by_day[day],
            s=150,
            edgecolors=edgecolors,
            linewidths=1.5,
            alpha=0.95,
            zorder=3,
        )

    if HETEROGENEITY_AXIS_SCALE == "log":
        agreement = np.geomspace(lower, upper, 300)
    else:
        agreement = np.linspace(lower, upper, 300)

    reference_color = shu.get_color("reference")
    ax.plot(
        agreement,
        agreement,
        "--",
        color=reference_color,
        lw=2.0,
        zorder=2,
    )

    if ANNOTATE_CASE_NAMES:
        for _, row in data.iterrows():
            ax.annotate(
                row["case_3d"],
                (row["x_plot"], row["y_plot"]),
                xytext=(5, 4),
                textcoords="offset points",
                fontsize=10,
            )

    ax.set_xscale(HETEROGENEITY_AXIS_SCALE)
    ax.set_yscale(HETEROGENEITY_AXIS_SCALE)
    ax.set_xlim(lower, upper)
    ax.set_ylim(lower, upper)
    ax.set_aspect("equal", adjustable="box")

    ax.set_xlabel(r"2D spatial heterogeneity $H_\nu$")
    ax.set_ylabel(r"3D spatial heterogeneity $H_\nu$")
    ax.set_title(
        "Cross-case 2D-3D particle-heterogeneity comparison"
        f" ({particle_label})"
    )

    ax.grid(True, which="major", alpha=0.35)
    ax.grid(True, which="minor", alpha=0.12)

    legend_handles = [
        Line2D(
            [0], [0],
            marker=marker_by_day[day],
            linestyle="none",
            markerfacecolor="0.55",
            markeredgecolor="white",
            markersize=10,
            label=f"T={day:g} d",
        )
        for day in days
    ]

    ax.legend(
        handles=legend_handles,
        loc="lower right",
        frameon=True,
        framealpha=0.95,
        title="Selected snapshots",
    )

    mappable = ScalarMappable(norm=norm, cmap=cmap)
    mappable.set_array([])
    colorbar = fig.colorbar(
        mappable,
        ax=ax,
        pad=0.025,
        fraction=0.05,
    )
    colorbar.set_label(
        COLOR_LABELS.get(COLOR_VARIABLE, COLOR_VARIABLE)
    )

    fig.tight_layout(rect=(0.0, 0.045, 1.0, 1.0))

    if SAVE_FIGURES:
        output_file = (
            OUTPUT_DIR
            / (
                "particle_heterogeneity_2D_vs_3D_"
                f"{particle_slug(particle_key)}_"
                f"{COLOR_VARIABLE.replace('.', '_')}.png"
            )
        )
        fig.savefig(
            output_file,
            dpi=getattr(ptheme, "SAVE_DPI", 300),
            bbox_inches="tight",
            facecolor="white",
            transparent=False,
        )
        print(f"Saved: {output_file}")

    plt.show()
    return fig, ax


figures = {}
for particle in particle_catalog.itertuples(index=False):
    figures[particle.particle_key] = plot_cross_case_heterogeneity(
        comparison_df,
        particle_key=particle.particle_key,
        particle_label=particle.particle_label,
    )
